# Week 2 — TOU Tariff Cost Matrix for Plant 1 and Plant 2

## Purpose
Quantify a **scenario estimate** of the value of storing measured solar generation and discharging it during peak tariff hours, matching the project brief.

## Historical tariff benchmark
APERC Retail Supply Tariff Order FY 2020–21, **HT Industry (General), assumed 11 kV**:
- Peak: 06:00–10:00 and 18:00–22:00 — ₹7.30/kWh
- Normal/base: 10:00–18:00 — ₹6.70/kWh
- Off-peak: 22:00–06:00 — ₹5.30/kWh

Official APERC tariff-order listing: https://aperc.gov.in/page/Tariff_Orders?currentpage=2
The order's industrial schedule is also mirrored in the public tariff summary: https://www.scribd.com/document/461689374/Electricity-Retail-Supply-Tariff-Andhra-Pradesh-2020-21

**Applicability caveat:** plant data do not specify a consumer tariff category or voltage. 11 kV HT Industry (General) is an explicit scenario assumption, not a claim about the plants' actual bills.

## Battery assumptions (editable)
- Capacity: 100 kWh
- Charge/discharge power limit: 25 kW
- Round-trip efficiency: 90%
- Charge from measured solar output during 10:00–16:00
- Discharge during evening peak 18:00–22:00
- Battery state of charge resets daily for a simple daily-cycle scenario

**Important:** generation CSVs do not contain plant consumption, grid imports, export compensation, or battery meter data. Therefore the matrix estimates tariff arbitrage value only; it cannot claim actual bill savings.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PLANT1_CSV = "/content/Plant_1_Generation_Data.csv"
PLANT2_CSV = "/content/Plant_2_Generation_Data.csv"

# In Colab, upload the two original generation CSVs if these paths don't exist.
if not Path(PLANT1_CSV).exists() or not Path(PLANT2_CSV).exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        names = list(uploaded)
        for name in names:
            low = name.lower()
            if "plant_1_generation" in low:
                PLANT1_CSV = name
            elif "plant_2_generation" in low:
                PLANT2_CSV = name
    except ImportError as e:
        raise FileNotFoundError("Set PLANT1_CSV and PLANT2_CSV to your local generation CSV paths.") from e

def load_plant(path, label, dayfirst=False):
    raw = pd.read_csv(path)
    raw["timestamp"] = pd.to_datetime(raw["DATE_TIME"], dayfirst=dayfirst, errors="coerce")
    raw["AC_POWER"] = pd.to_numeric(raw["AC_POWER"], errors="coerce")
    raw = raw.dropna(subset=["timestamp"])
    return (raw.groupby("timestamp", as_index=False)["AC_POWER"].sum()
            .rename(columns={"AC_POWER": "solar_generation_kw"})
            .assign(plant=label)
            .sort_values("timestamp"))

p1 = load_plant(PLANT1_CSV, "Plant 1", dayfirst=True)
p2 = load_plant(PLANT2_CSV, "Plant 2", dayfirst=False)
start, end = pd.Timestamp("2020-05-15 00:00"), pd.Timestamp("2020-06-17 23:45")
grid = pd.date_range(start, end, freq="15min")
print("Plant 1 rows:", len(p1), "| Plant 2 rows:", len(p2))


In [ ]:
# APERC FY2020-21 HT Industry (General), assumed 11 kV tariff benchmark.
RATES = {"Normal": 6.70, "Peak": 7.30, "Off-peak": 5.30}
INTERVAL_HOURS = 0.25
BATTERY_CAPACITY_KWH = 100.0
BATTERY_POWER_KW = 25.0
ROUND_TRIP_EFFICIENCY = 0.90
CHARGE_EFF = DISCHARGE_EFF = np.sqrt(ROUND_TRIP_EFFICIENCY)

def period_for_hour(hour):
    if 6 <= hour < 10 or 18 <= hour < 22:
        return "Peak"
    if hour >= 22 or hour < 6:
        return "Off-peak"
    return "Normal"

def build_matrix(plant_df, plant_name):
    generation = plant_df.set_index("timestamp")["solar_generation_kw"].reindex(grid)
    result = pd.DataFrame({"timestamp": grid, "solar_generation_kw": generation.values})
    result["plant"] = plant_name
    result["tariff_period"] = result["timestamp"].dt.hour.map(period_for_hour)
    result["tariff_rate_inr_per_kwh"] = result["tariff_period"].map(RATES)
    result["solar_energy_kwh"] = result["solar_generation_kw"] * INTERVAL_HOURS

    soc = 0.0
    fields = {k: [] for k in [
        "battery_soc_kwh", "battery_charge_kw", "battery_discharge_kw",
        "battery_charge_energy_kwh", "battery_discharge_energy_kwh",
        "gross_peak_discharge_value_inr", "normal_tariff_energy_reference_cost_inr",
        "incremental_tou_arbitrage_value_inr"
    ]}
    for _, row in result.iterrows():
        ts, power = row["timestamp"], row["solar_generation_kw"]
        if ts.hour == 0 and ts.minute == 0:
            soc = 0.0
        if pd.isna(power):
            for k in fields: fields[k].append(np.nan)
            continue
        charge_kw = discharge_kw = 0.0
        if 10 <= ts.hour < 16:
            charge_kw = min(max(power, 0), BATTERY_POWER_KW,
                            max(0, (BATTERY_CAPACITY_KWH - soc) /
                                (INTERVAL_HOURS * CHARGE_EFF)))
            soc += charge_kw * INTERVAL_HOURS * CHARGE_EFF
        elif 18 <= ts.hour < 22:
            discharge_kw = min(BATTERY_POWER_KW, soc * DISCHARGE_EFF / INTERVAL_HOURS)
            discharge_kw = max(0, discharge_kw)
            soc -= discharge_kw * INTERVAL_HOURS / DISCHARGE_EFF
        ch_kwh, dis_kwh = charge_kw * INTERVAL_HOURS, discharge_kw * INTERVAL_HOURS
        gross = dis_kwh * RATES["Peak"]
        ref_cost = (dis_kwh / ROUND_TRIP_EFF) * RATES["Normal"] if dis_kwh else 0.0
        values = [soc, charge_kw, discharge_kw, ch_kwh, dis_kwh, gross, ref_cost, gross - ref_cost]
        for k, v in zip(fields, values): fields[k].append(v)
    for k, values in fields.items(): result[k] = values
    result["data_note"] = "Measured solar; battery and tariff value are scenario assumptions, not actual bill savings"
    return result

m1 = build_matrix(p1, "Plant 1")
m2 = build_matrix(p2, "Plant 2")
combined_input = pd.concat([p1, p2]).groupby("timestamp", as_index=False)["solar_generation_kw"].sum()
mc = build_matrix(combined_input, "Combined Plants")
matrix = pd.concat([m1, m2, mc], ignore_index=True)
matrix.to_csv("Week2_TOU_Tariff_Cost_Matrix_Plant1_Plant2_2020.csv", index=False)


In [ ]:
def summarize(m):
    return {
        "plant": m["plant"].iloc[0],
        "intervals": len(m),
        "missing_solar_intervals": int(m["solar_generation_kw"].isna().sum()),
        "measured_solar_energy_kwh": m["solar_energy_kwh"].sum(min_count=1),
        "battery_charge_input_kwh": m["battery_charge_energy_kwh"].sum(min_count=1),
        "battery_discharge_delivered_kwh": m["battery_discharge_energy_kwh"].sum(min_count=1),
        "gross_peak_discharge_value_inr": m["gross_peak_discharge_value_inr"].sum(min_count=1),
        "normal_tariff_reference_cost_inr": m["normal_tariff_energy_reference_cost_inr"].sum(min_count=1),
        "incremental_tou_arbitrage_value_inr": m["incremental_tou_arbitrage_value_inr"].sum(min_count=1),
    }

summary = pd.DataFrame([summarize(m1), summarize(m2), summarize(mc)])
display(summary.round(2))
summary.to_csv("Week2_TOU_Tariff_Summary_Plant1_Plant2.csv", index=False)


## How to interpret results
- `gross_peak_discharge_value_inr` is the value of delivered battery energy at the peak rate.
- `normal_tariff_reference_cost_inr` values the charging energy at the normal-rate benchmark, adjusted for assumed round-trip losses.
- `incremental_tou_arbitrage_value_inr` is the difference between those values.
- This is **not actual bill savings** because there is no plant-load or grid-import dataset, and no actual tariff contract is supplied.
- The combined plants result is a separate aggregate view; do not add it to the individual plant results.
